# WeTe PPD Notebook

Unified `.mat` outputs (`beta`, `train_theta`, `test_theta`).
Uses official WeTe source if available, otherwise a fallback implementation.


## 1) Imports


In [ ]:
import os
import random
from pathlib import Path

import numpy as np
import scipy
import scipy.io
import scipy.sparse as sp
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.cluster import KMeans
from sklearn import metrics

print('Python:', __import__('sys').version.split()[0])
print('NumPy:', np.__version__)
print('SciPy:', scipy.__version__)
print('Torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())


## 2) Dataset discovery


In [ ]:
TARGET_DATASETS = ['20NG', 'AGNews', 'IMDB', 'YahooAnswer']
K_LIST = [20, 50, 100]
RANDOM_SEED = 42

REQUIRED_DATA_FILES = [
    'train_bow.npz', 'test_bow.npz', 'train_labels.txt', 'test_labels.txt', 'vocab.txt', 'word_embeddings.npz'
]

IS_KAGGLE = Path('/kaggle').exists()


def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / 'ECTRM_source' / 'ECRTM' / 'data').exists():
            return candidate
    return start


if IS_KAGGLE:
    PROJECT_ROOT = Path('/kaggle/working')
    INPUT_ROOT = Path('/kaggle/input')
else:
    PROJECT_ROOT = find_project_root(Path.cwd())
    INPUT_ROOT = PROJECT_ROOT


def has_required_files(path: Path) -> bool:
    return path.exists() and path.is_dir() and all((path / f).exists() for f in REQUIRED_DATA_FILES)


def guess_dataset_name(path: Path):
    s = str(path).lower()
    if '20ng' in s or '20news' in s:
        return '20NG'
    if 'agnews' in s or 'ag_news' in s:
        return 'AGNews'
    if 'yahoo' in s:
        return 'YahooAnswer'
    if 'imdb' in s:
        return 'IMDB'
    return None


def iter_candidate_dirs(base: Path, max_depth: int = 5):
    if not base.exists():
        return
    base_depth = len(base.parts)
    for root, dirs, _ in os.walk(base):
        root_path = Path(root)
        depth = len(root_path.parts) - base_depth
        if depth > max_depth:
            dirs[:] = []
            continue
        yield root_path


def discover_dataset_dirs():
    found = {}

    local_data_root = PROJECT_ROOT / 'ECTRM_source' / 'ECRTM' / 'data'
    for ds in TARGET_DATASETS:
        p = local_data_root / ds
        if has_required_files(p):
            found[ds] = p

    scan_roots = [Path('/kaggle/input'), PROJECT_ROOT, INPUT_ROOT] if IS_KAGGLE else [PROJECT_ROOT, INPUT_ROOT]
    seen = set()
    for scan_root in scan_roots:
        for cand in iter_candidate_dirs(scan_root):
            cs = str(cand)
            if cs in seen:
                continue
            seen.add(cs)
            if not has_required_files(cand):
                continue
            ds = guess_dataset_name(cand)
            if ds is not None and ds not in found:
                found[ds] = cand
    return found


DATASET_DIRS = discover_dataset_dirs()
OUTPUT_DIR = (Path('/kaggle/working') / 'Sortie_mat' / 'output' / 'kaggle' / 'WeTe') if IS_KAGGLE else (PROJECT_ROOT / 'Sortie_mat' / 'WeTe')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print('PROJECT_ROOT:', PROJECT_ROOT)
print('OUTPUT_DIR:', OUTPUT_DIR)
print('Resolved datasets:')
for ds in TARGET_DATASETS:
    print('-', ds, DATASET_DIRS.get(ds))


## 3) Utils


In [ ]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def load_vocab(path: Path):
    with open(path, 'r', encoding='utf-8') as f:
        return [line.strip() for line in f if line.strip()]


def load_bow_csr(path: Path):
    return sp.load_npz(path).astype(np.float32).tocsr()


def load_word_embeddings(path: Path, vocab_size=None):
    try:
        arr = sp.load_npz(path).astype(np.float32).toarray()
        if vocab_size is not None and arr.shape[0] != vocab_size and arr.shape[1] == vocab_size:
            arr = arr.T
        return arr
    except Exception:
        data = np.load(path)
        if isinstance(data, np.lib.npyio.NpzFile):
            arr = data[list(data.keys())[0]]
        else:
            arr = data
        arr = np.asarray(arr, dtype=np.float32)
        if vocab_size is not None:
            if arr.ndim == 1:
                emb_dim = arr.size // vocab_size
                arr = arr[: vocab_size * emb_dim].reshape(vocab_size, emb_dim)
            elif arr.shape[0] != vocab_size and arr.shape[1] == vocab_size:
                arr = arr.T
        return arr.astype(np.float32)


class BowDataset(Dataset):
    def __init__(self, bow_csr, labels):
        self.bow = bow_csr.tocsr()
        self.labels = labels.astype(np.int32)

    def __len__(self):
        return self.bow.shape[0]

    def __getitem__(self, idx):
        row = self.bow[idx].toarray().astype(np.float32).squeeze(0)
        return torch.from_numpy(row), int(self.labels[idx])


## 4) Backend selection


In [ ]:
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('DEVICE:', DEVICE)

WETE_DEFAULTS = {
    'epochs': 300,
    'batch_size': 500,
    'lr': 1e-2,
    'beta': 0.5,
    'epsilon': 1.0,
    'hidden_size': 800,
}

wete_repo = PROJECT_ROOT / 'WeTe_source' / 'WeTe'
OFFICIAL_WETE_AVAILABLE = (wete_repo / 'model.py').exists() and (wete_repo / 'Utils.py').exists()
print('OFFICIAL_WETE_AVAILABLE:', OFFICIAL_WETE_AVAILABLE)

if OFFICIAL_WETE_AVAILABLE:
    import sys
    if str(wete_repo) not in sys.path:
        sys.path.insert(0, str(wete_repo))
    from model import WeTe as OfficialWeTe
    from Utils import to_list


## 5) Models


In [ ]:
class WeTeFallback(nn.Module):
    def __init__(self, vocab_size, num_topics, hidden_size=800, dropout=0.2):
        super().__init__()
        self.num_topics = num_topics

        self.encoder = nn.Sequential(
            nn.Linear(vocab_size, hidden_size),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_size, hidden_size),
            nn.ReLU(),
            nn.Dropout(dropout),
        )
        self.fc_mu = nn.Linear(hidden_size, num_topics)
        self.fc_logvar = nn.Linear(hidden_size, num_topics)
        self.beta_logits = nn.Parameter(torch.randn(num_topics, vocab_size) * 0.01)

    def _theta(self, x):
        h = self.encoder(x)
        mu = self.fc_mu(h)
        logvar = self.fc_logvar(h)
        if self.training:
            eps = torch.randn_like(mu)
            z = mu + eps * torch.exp(0.5 * logvar)
        else:
            z = mu
        theta = torch.softmax(z, dim=1)
        return theta, mu, logvar

    def get_beta(self):
        return torch.softmax(self.beta_logits, dim=1)

    def get_theta(self, x):
        self.eval()
        with torch.no_grad():
            theta, _, _ = self._theta(x)
        return theta

    def forward(self, x, kl_weight=0.2, div_weight=0.5):
        theta, mu, logvar = self._theta(x)
        beta = self.get_beta()
        recon = torch.matmul(theta, beta)

        recon_loss = -(x * torch.log(recon + 1e-10)).sum(1).mean()
        kl = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp(), dim=1).mean()

        b = torch.nn.functional.normalize(beta, p=2, dim=1)
        corr = torch.mm(b, b.t())
        eye = torch.eye(corr.shape[0], device=corr.device)
        div = torch.mean((corr - eye) ** 2)

        return recon_loss + kl_weight * kl + div_weight * div


def build_model(K, vocab, word_emb, cfg, seed=42):
    if OFFICIAL_WETE_AVAILABLE:
        from types import SimpleNamespace
        args = SimpleNamespace(
            K=K,
            vocsize=len(vocab),
            embedding_dim=int(word_emb.shape[1]),
            beta=float(cfg['beta']),
            epsilon=float(cfg['epsilon']),
            init_alpha=False,
            device='cuda:0' if torch.cuda.is_available() else 'cpu',
            glove=None,
        )
        model = OfficialWeTe(args, voc=vocab).to(DEVICE)
        model.word_layer = nn.Embedding.from_pretrained(torch.from_numpy(word_emb).float(), freeze=False).to(DEVICE)
        km = KMeans(n_clusters=K, random_state=seed, n_init='auto')
        centers = km.fit(word_emb).cluster_centers_.astype(np.float32)
        model.topic_layer = nn.Embedding.from_pretrained(torch.from_numpy(centers), freeze=False).to(DEVICE)
        model.update_embeddings()
        return model

    return WeTeFallback(vocab_size=len(vocab), num_topics=K, hidden_size=int(cfg['hidden_size'])).to(DEVICE)


def train_model(model, train_loader, cfg, seed=42):
    set_seed(seed)
    optimizer = torch.optim.Adam([p for p in model.parameters() if p.requires_grad], lr=float(cfg['lr']), weight_decay=1e-3)

    for epoch in range(int(cfg['epochs'])):
        model.train()
        losses = []
        for bow, _ in train_loader:
            bow = bow.to(DEVICE).float()
            optimizer.zero_grad()
            if OFFICIAL_WETE_AVAILABLE:
                token_lists = to_list(bow.long(), device=('cuda:0' if torch.cuda.is_available() else 'cpu'))
                loss, _, _, _, _ = model(token_lists, bow)
            else:
                loss = model(bow)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=20.0)
            optimizer.step()
            losses.append(float(loss.item()))

        if (epoch + 1) % 10 == 0 or epoch == 0:
            print(f'Epoch {epoch+1:03d}/{cfg["epochs"]} - loss={np.mean(losses):.4f}')


def infer_theta(model, loader):
    model.eval()
    out = []
    with torch.no_grad():
        for bow, _ in loader:
            bow = bow.to(DEVICE).float()
            if OFFICIAL_WETE_AVAILABLE:
                theta = torch.softmax(model.InferNet(bow), dim=1)
            else:
                theta = model.get_theta(bow)
            out.append(theta.cpu().numpy().astype(np.float32))
    return np.vstack(out)


def get_beta_np(model):
    model.eval()
    with torch.no_grad():
        if OFFICIAL_WETE_AVAILABLE:
            model.update_embeddings()
            beta = model.cal_phi().cpu().numpy().astype(np.float32).T
        else:
            beta = model.get_beta().cpu().numpy().astype(np.float32)
    return beta


## 6) Training and export


In [ ]:
def run_one_wete(dataset, K, seed=42, cfg=None):
    cfg = dict(WETE_DEFAULTS if cfg is None else cfg)

    if dataset not in DATASET_DIRS:
        raise KeyError(f'Dataset {dataset} not found')

    data_dir = DATASET_DIRS[dataset]
    train_csr = load_bow_csr(data_dir / 'train_bow.npz')
    test_csr = load_bow_csr(data_dir / 'test_bow.npz')
    train_labels = np.loadtxt(data_dir / 'train_labels.txt', dtype=np.int32)
    test_labels = np.loadtxt(data_dir / 'test_labels.txt', dtype=np.int32)
    vocab = load_vocab(data_dir / 'vocab.txt')
    word_emb = load_word_embeddings(data_dir / 'word_embeddings.npz', vocab_size=train_csr.shape[1])

    train_ds = BowDataset(train_csr, train_labels)
    test_ds = BowDataset(test_csr, test_labels)
    train_loader = DataLoader(train_ds, batch_size=int(cfg['batch_size']), shuffle=True, num_workers=0)
    test_loader = DataLoader(test_ds, batch_size=int(cfg['batch_size']), shuffle=False, num_workers=0)

    print(f'\n=== WeTe {dataset} K={K} | backend={"official" if OFFICIAL_WETE_AVAILABLE else "fallback"} ===')
    model = build_model(K, vocab, word_emb, cfg, seed=seed)
    train_model(model, train_loader, cfg, seed=seed)

    beta = get_beta_np(model)
    train_theta = infer_theta(model, train_loader)
    test_theta = infer_theta(model, test_loader)

    ds_out = OUTPUT_DIR / dataset
    ds_out.mkdir(parents=True, exist_ok=True)
    out_path = ds_out / f'{dataset}_WeTe_K{K}_seed{seed}.mat'

    scipy.io.savemat(str(out_path), {'beta': beta, 'train_theta': train_theta, 'test_theta': test_theta, 'traintheta': train_theta, 'testtheta': test_theta})
    print('Saved:', out_path)
    return out_path


## 7) Run and metrics


In [ ]:
import pandas as pd


def topic_diversity_from_beta(beta, topn=25):
    all_words = []
    for k in range(beta.shape[0]):
        top_idx = np.argsort(beta[k])[::-1][:topn]
        all_words.extend(top_idx.tolist())
    return len(set(all_words)) / max(1, len(all_words))


def purity_score(y_true, y_pred):
    contingency_matrix = metrics.cluster.contingency_matrix(y_true, y_pred)
    return np.sum(np.amax(contingency_matrix, axis=0)) / np.sum(contingency_matrix)


for dataset in TARGET_DATASETS:
    if dataset not in DATASET_DIRS:
        print('SKIP missing dataset:', dataset)
        continue
    for K in K_LIST:
        run_one_wete(dataset, K=K, seed=RANDOM_SEED)

rows = []
TOPN = 15
for dataset in TARGET_DATASETS:
    if dataset not in DATASET_DIRS:
        continue

    labels = np.loadtxt(DATASET_DIRS[dataset] / 'test_labels.txt', dtype=np.int32)
    vocab = load_vocab(DATASET_DIRS[dataset] / 'vocab.txt')
    ds_out = OUTPUT_DIR / dataset

    for K in K_LIST:
        mat_path = ds_out / f'{dataset}_WeTe_K{K}_seed{RANDOM_SEED}.mat'
        if not mat_path.exists():
            continue

        loaded = scipy.io.loadmat(str(mat_path))
        beta = loaded['beta']
        test_theta = loaded['test_theta']

        n_eval = min(len(labels), test_theta.shape[0])
        y_true = labels[:n_eval]
        y_pred = np.argmax(test_theta[:n_eval], axis=1)

        rows.append({
            'model': 'WeTe',
            'dataset': dataset,
            'K': K,
            'Purity': float(purity_score(y_true, y_pred)),
            'NMI': float(metrics.cluster.normalized_mutual_info_score(y_true, y_pred)),
            'TD_top25': float(topic_diversity_from_beta(beta, topn=25)),
        })

        txt_path = ds_out / f'topics_for_cv_{dataset}_WeTe_K{K}_seed{RANDOM_SEED}.txt'
        with open(txt_path, 'w', encoding='utf-8') as f:
            for k in range(beta.shape[0]):
                top_idx = np.argsort(beta[k])[::-1][:TOPN]
                f.write(' '.join(vocab[i] for i in top_idx) + '\n')
        print('Saved:', txt_path)

if rows:
    df = pd.DataFrame(rows).sort_values(['dataset', 'K']).reset_index(drop=True)
    display(df)
    csv_path = OUTPUT_DIR / 'WeTe_metrics_TD_Purity_NMI.csv'
    df.to_csv(csv_path, index=False)
    print('Saved:', csv_path)
else:
    print('No WeTe .mat results found yet')
